---
# 0. Setup

In [ ]:
from pathlib import Path

import holidays as hols
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

# Notebooks live in EDA/, data lives in <repo root>/data/ (git-ignored -- never commit it).
DATA_DIR = Path('.') / 'data'



---
# 1.  Airport Info Joining (Airport location/specs to county/CBSA)



In [ ]:
import json

from shapely.geometry import shape

# 1. Load County Boundaries CSV
counties_raw = pd.read_csv(
    DATA_DIR /"us-county-boundaries.csv",  
    sep=";",
    dtype={"GEOID": str, "STATEFP": str, "COUNTYFP": str, "CBSAFP": str},
)

# Parse the GeoJSON strings in Geo Shape into Shapely polygons
counties_raw["geometry"] = counties_raw["Geo Shape"].apply(
    lambda x: shape(json.loads(x)) if pd.notnull(x) else None
)

gdf_counties = gpd.GeoDataFrame(
    counties_raw[["GEOID", "CBSAFP", "STATE_NAME", "geometry"]],
    geometry="geometry",
    crs="EPSG:4326",
)

# 2. Load Airport Coordinates CSV
airports = pd.read_csv(DATA_DIR / "AA_AIRPORT_SIMPLE.csv")

gdf_airports = gpd.GeoDataFrame(
    airports,
    geometry=gpd.points_from_xy(airports["LNGTD"], airports["LAT"]),
    crs="EPSG:4326",
)

# 3. Spatial Join: Assign GEOID (County_FIPS) and CBSAFP to every US/PR airport
dim_airports = gpd.sjoin(
    gdf_airports, gdf_counties, how="left", predicate="within"
).drop(columns=["geometry", "index_right"])

dim_airports = dim_airports.rename(
    columns={"GEOID": "County_FIPS", "CBSAFP": "CBSA_Code"}
)

assert dim_airports["AIRPRT_CD"].is_unique, "Duplicate airport codes found!"
print(f"Total airports: {len(dim_airports)}")
print(f"Matched to US/PR County: {dim_airports['County_FIPS'].notna().sum()}")
print(f"Matched to a CBSA: {dim_airports['CBSA_Code'].notna().sum()}")




In [ ]:
dim_airports.sample(10)
dim_airports.isnull().value_counts()


In [ ]:

dim_airports.drop(columns=['STATE_NAME'], axis=1, inplace=True)
dim_airports.drop(columns=['CITY_METRO_IATA_CD'],axis=1,inplace=True)
dim_airports.drop(columns=['AIRPRT_NM','STATE_PROVNC_CD','WRLD_AREA_DOT_CD'], inplace=True)
dim_airports.fillna("MISSING",inplace=True)
dim_airports.head()


In [ ]:
pd.set_option('display.float_format', '{:,.6f}'.format)
dim_airports.rename(columns={'County_FIPS':'CNTY_FIPS','CBSA_Code':'CBSA_CD'}, inplace=True)
dim_airports.head()

In [ ]:
dim_airports.to_csv(DATA_DIR / "AIRPORT_V1.csv", index=False)

---
# 2. Forecast info (PAX, REV) + Calendar features

In [ ]:


def extend_route_spine(
    df: pd.DataFrame,
    end_year: int,
    end_month: int,
    route_cols: list = ["ORG_AIRPRT", "DES_AIRPRT"],
    year_col: str = "DEP_YEAR",
    month_col: str = "DEP_MONTH",
    fill_zero_cols: list = ["OD_PAX_LOG", "OD_REV_USD_LOG"],
) -> pd.DataFrame:
    """
    Expands a route-month dataframe so every unique route has a continuous row
    from the earliest [Year, Month] in the data up to (end_year, end_month).
    
    Historical gaps (<= max date in df) in `fill_zero_cols` are filled with 0;
    future forecast months (> max date in df) are left as NaN.
    """
    df = df.copy()
    df['year'] = df[year_col]
    df['month'] = df[month_col]
    df.drop(columns=[year_col, month_col], inplace=True)
    year_col = 'year'
    month_col = 'month'
    # 1. Detect the earliest and latest [Year, Month] already in the dataframe
    existing_dates = pd.to_datetime(
        df[[year_col, month_col]].assign(DAY=1)
    )
    min_date = existing_dates.min()
    max_hist_date = existing_dates.max()
    target_end_date = pd.Timestamp(year=end_year, month=end_month, day=1)

    if target_end_date < min_date:
        raise ValueError(f"Target end date ({target_end_date:%Y-%m}) is earlier than min date ({min_date:%Y-%m}).")

    # 2. Extract unique routes and generate the complete monthly date grid
    unique_routes = df[route_cols].drop_duplicates()
    all_dates = pd.date_range(start=min_date, end=target_end_date, freq="MS")
    
    date_grid = pd.DataFrame({
        year_col: all_dates.year,
        month_col: all_dates.month,
        "_temp_date": all_dates,
    })

    # 3. Cross-join routes x dates, then left-join original metrics back
    expanded = unique_routes.merge(date_grid, how="cross")
    expanded = expanded.merge(df, on=route_cols + [year_col, month_col], how="left")

    # 4. Fill missing historical months (<= T) with 0; keep future rows (> T) as NaN
    valid_fill_cols = [c for c in fill_zero_cols if c in expanded.columns]
    if valid_fill_cols:
        hist_mask = expanded["_temp_date"] <= max_hist_date
        expanded.loc[hist_mask, valid_fill_cols] = (
            expanded.loc[hist_mask, valid_fill_cols].fillna(0)
        )

    # 5. Sort cleanly and drop helper date column
    expanded = (
        expanded.sort_values(route_cols + [year_col, month_col])
        .drop(columns=["_temp_date"])
        .reset_index(drop=True)
    )
    expanded['ROUTE'] = expanded[route_cols[0]].astype(str) + '-'+expanded[route_cols[1]].astype(str)
    

    return expanded


In [ ]:
base_df = pd.read_csv(DATA_DIR / "made/AA_join.csv")
base_df = base_df[['DEP_MONTH','DEP_YEAR','ORG_AIRPRT','DES_AIRPRT','ROUTE','OD_REV_USD_LOG','OD_PAX_LOG']]
base_df.tail()


In [ ]:
# Auto-detect your exact last valid historical date T from raw Passenger/Revenue CSV
full_spine = base_df.copy()
full_spine['ROUTE'] = full_spine['ORG_AIRPRT'].astype(str) + '-' + full_spine['DES_AIRPRT'].astype(str)
full_spine['DATE'] = pd.to_datetime(
    dict(year=full_spine["DEP_YEAR"], month=full_spine["DEP_MONTH"], day=1)
)
raw_max_date = pd.to_datetime(
    dict(year=full_spine["DEP_YEAR"], month=full_spine["DEP_MONTH"], day=1)
).max()
print("Exact historical cutoff T:", raw_max_date.strftime("%Y-%m"))

# fill 0 on or before max date and leave all after as NaN
hist_mask = full_spine["DATE"] <= raw_max_date
full_spine.loc[hist_mask, ["OD_PAX_LOG", "OD_REV_USD_LOG"]] = (
    full_spine.loc[hist_mask, ["OD_PAX_LOG", "OD_REV_USD_LOG"]].fillna(0)
)


In [ ]:
base_df = extend_route_spine(full_spine,2028,7)
base_df.drop(columns=['DATE'],inplace=True)
base_df.rename(columns={'year':'DEP_YEAR','month':'DEP_MONTH'},inplace=True)
base_df.sample(5)


In [ ]:
base_df.isnull().value_counts()

In [ ]:
base_df.rename(columns={'year':'DEP_YEAR','month':'DEP_MONTH'}, inplace=True)
base_df.groupby(['DEP_YEAR','DEP_MONTH']).agg({'OD_PAX_LOG':'sum','OD_REV_USD_LOG':'sum'}).reset_index().sort_values(['DEP_YEAR','DEP_MONTH'],ascending=True).iloc[-30:]

In [ ]:
calendar = pd.read_csv(DATA_DIR / 'calendar_features.csv')
calendar.drop(columns=['is_summer_break','is_winter_holiday'],inplace=True)
calendar.sample(5)

In [ ]:
base_df.isnull().value_counts()

In [ ]:
full_spine = base_df.join(calendar.set_index(['flight_month','flight_year']), on=['DEP_MONTH','DEP_YEAR'], how='left')
full_spine.sort_values(by=['DEP_YEAR','DEP_MONTH']).tail()

# 3. T100 + Base DF

In [ ]:
base_df = full_spine.copy()
base_df.head()

In [ ]:
T100_2017 = pd.read_csv(DATA_DIR / 'T100_2017.csv')
T100_2018 = pd.read_csv(DATA_DIR / 'T100_2018.csv')
T100_2019 = pd.read_csv(DATA_DIR / 'T100_2019.csv')
T100_2020 = pd.read_csv(DATA_DIR / 'T100_2020.csv')
T100_2021 = pd.read_csv(DATA_DIR / 'T100_2021.csv')
T100_2022 = pd.read_csv(DATA_DIR / 'T100_2022.csv')
T100_2023 = pd.read_csv(DATA_DIR / 'T100_2023.csv')
T100_2024 = pd.read_csv(DATA_DIR / 'T100_2024.csv')
T100_2025 = pd.read_csv(DATA_DIR / 'T100_2025.csv')
T100_2026 = pd.read_csv(DATA_DIR / 'T100_2026.csv')
T100 = pd.concat([T100_2017,T100_2018, T100_2019, T100_2020, T100_2021, T100_2022, T100_2023, T100_2024, T100_2025, T100_2026], ignore_index=True)


In [ ]:
T100.to_csv(DATA_DIR / 'T100.csv', index=False)


In [ ]:
T100.sort_values(by=['YEAR','MONTH']).tail()

In [ ]:

def create_grid_lags(
    df: pd.DataFrame,
    entity_cols: list,
    value_cols: list,
    lags: list = [12, 24],
    max_target_date: str = "2028-12-31",
    year_col: str = "Year",
    month_col: str = "Month",
    drop_raw: bool = True,
) -> pd.DataFrame:
    # Uses `df`, `year_col`, and `month_col` to build a proper timestamp
    work_df = df.copy()
    work_df["_date"] = pd.to_datetime(
        dict(year=work_df[year_col], month=work_df[month_col], day=1)
    )

    min_date = work_df["_date"].min()
    max_hist_date = work_df["_date"].max()
    # Uses `max_target_date` to set the end of the monthly calendar
    end_date = pd.to_datetime(max_target_date)

    # Uses `year_col` and `month_col` to build the complete monthly date grid
    all_dates = pd.date_range(start=min_date, end=end_date, freq="MS")
    date_grid = pd.DataFrame({
        year_col: all_dates.year,
        month_col: all_dates.month,
        "_date": all_dates,
    })

    # Initializes `grid` by cross-joining unique entities with every month in `date_grid`
    entities = work_df.loc[:, entity_cols].drop_duplicates()
    join_keys = list(entity_cols) + [year_col, month_col]
    
    grid = entities.merge(date_grid, how="cross").merge(
        work_df.drop(columns=["_date"]),
        on=join_keys,
        how="left",
    )

    # Fills missing historical months (<= max date in `df`) with 0 and keeps future rows (> T) as NaN
    hist_mask = grid["_date"] <= max_hist_date
    grid.loc[hist_mask, value_cols] = grid.loc[hist_mask, value_cols].fillna(0)

    # Sorts `grid` chronologically and shifts `value_cols` by each lag in `lags`
    sort_keys = list(entity_cols) + ["_date"]
    grid = grid.sort_values(sort_keys).reset_index(drop=True)

    for col in value_cols:
        for l in lags:
            grid[f"{col}_lag_{l}"] = grid.groupby(entity_cols)[col].shift(l)

    cols_to_drop = ["_date"] + (list(value_cols) if drop_raw else list())
    return grid.drop(columns=cols_to_drop)


In [ ]:
def left_join_feature_table(
    spine: pd.DataFrame,
    feature_df: pd.DataFrame,
    left_keys: list,
    right_keys: list = None,
    prefixes: list = None,
) -> pd.DataFrame:
  """
    Left-joins `feature_df` onto `spine` while strictly preserving row count.
    - For a single join : leave `prefixes=None`.
    - For a double join : pass `left_keys=[["Origin", "Year", "Month"], ["Destination", "Year", "Month"]]`
      and `prefixes=["ORG_", "DES_"]`.
    """
  rows_before = len(spine)
  right_keys = right_keys or left_keys

  # Case A: Single Left-Join
  if prefixes is None:
    spine = spine.merge(
        feature_df,
        left_on=left_keys,
        right_on=right_keys,
        how="left",
    )
    # Drop duplicate right-hand join keys if names differed
    extra_keys = [k for k in right_keys if k not in left_keys]
    if extra_keys:
      spine = spine.drop(columns=extra_keys)

  # Case B: Multi/Double Left-Join 
    for l_keys, prefix in zip(left_keys, prefixes):
      prefixed_df = feature_df.add_prefix(prefix)
      prefixed_r_keys = [f"{prefix}{k}" for k in right_keys]

      spine = spine.merge(
          prefixed_df,
          left_on=l_keys,
          right_on=prefixed_r_keys,
          how="left",
      ).drop(columns=prefixed_r_keys)

  assert (
      len(spine) == rows_before
  ), f"Row count exploded! {rows_before} -> {len(spine)}"
  return spine


In [ ]:

t100 = T100.copy()
target_carrier = "AA"  
t100["route"] = (
    t100["ORIGIN"].astype(str).str.strip()
    + "-"
    + t100["DEST"].astype(str).str.strip()
)

# Aggregate Total Market Volume (All Carriers)
market_t100 = t100.groupby(["route", "YEAR", "MONTH"], as_index=False).agg(
    T100_Market_Pax=("PASSENGERS", "sum"),
    T100_Market_Freight=("FREIGHT", "sum"),
    T100_Carrier_Count=("UNIQUE_CARRIER", "nunique"),
)

# Aggregate Target Carrier Volume 
carrier_t100 = (
    t100[t100["UNIQUE_CARRIER"] == target_carrier]
    .groupby(["route", "YEAR", "MONTH"], as_index=False)
    .agg(T100_Target_Pax=("PASSENGERS", "sum"))
)

# Combine & Compute Competitive Market Share
t100_agg = market_t100.merge(
    carrier_t100, on=["route", "YEAR", "MONTH"], how="left"
).fillna({"T100_Target_Pax": 0})

t100_agg["T100_Target_Pax_Share"] = (
    t100_agg["T100_Target_Pax"] / t100_agg["T100_Market_Pax"].replace(0, pd.NA)
).fillna(0)

t100_agg = t100_agg.rename(columns={"YEAR": "Year", "MONTH": "Month"})

# Run your general lag function 
t100_lagged = create_grid_lags(
    df=t100_agg,
    entity_cols=["route"],
    value_cols=["T100_Market_Pax", "T100_Target_Pax_Share", "T100_Carrier_Count"],
    lags=(12, 24),
    max_target_date="2028-07-01",
    year_col="Year",
    month_col="Month",
    drop_raw=True,
)



In [ ]:
t100_lagged.sort_values(by=['Year','Month']).tail()

In [ ]:

full_spine = left_join_feature_table(
    spine=base_df,
    feature_df=t100_lagged,
    left_keys=["ROUTE", "DEP_YEAR", "DEP_MONTH"],
    right_keys=["route", "Year", "Month"]
)

In [ ]:
full_spine.head()

In [ ]:
full_spine.drop(columns=['route','T100_Market_Freight','T100_Target_Pax'],inplace=True, errors='ignore')
for col in full_spine.columns:
    print(full_spine[col].isnull().value_counts())

In [ ]:

unmatched_routes = full_spine[full_spine["T100_Market_Pax_lag_12"].isna()]["ROUTE"].unique()
print(f"Total unmatched routes: {len(unmatched_routes):,}")
print("First 20 unmatched routes:", unmatched_routes[100:150:3])



In [ ]:
full_spine.ROUTE.isnull().value_counts()
full_spine.head()

In [ ]:
full_spine.to_csv(DATA_DIR / "made/AA_T100.csv", index=False)

In [ ]:
t100_routes = set(t100["route"].unique())
matched_spine = full_spine[(full_spine["ROUTE"].isin(t100_routes)) & (full_spine["DEP_YEAR"] <= 2026)]
print("Null % on matched T-100 routes (2019-2026):", matched_spine["T100_Market_Pax_lag_12"].isna().mean() * 100)


In [ ]:
# Get the exact set of unique routes in spine vs. raw T-100 CSV
spine_routes = set(full_spine["ROUTE"].unique())
t100_routes = set(t100["route"].unique())

missing_from_t100 = spine_routes - t100_routes
in_both = spine_routes & t100_routes

print(f"Total spine routes: {len(spine_routes):,}")
print(f"Routes in T-100:    {len(in_both):,} ({len(in_both)/len(spine_routes)*100:.1f}%)")
print(f"Missing from T-100: {len(missing_from_t100):,} ({len(missing_from_t100)/len(spine_routes)*100:.1f}%)")

# Verify that among routes that ARE in T-100, there are 0.0% nulls across 2019-2026
matched_hist = full_spine[(full_spine["ROUTE"].isin(in_both)) & (full_spine["DEP_YEAR"] <= 2026)]
print("\nNull % on routes that exist in T-100 (2019-2026):")
print(matched_hist[["T100_Market_Pax_lag_12", "T100_Market_Pax_lag_24"]].isna().mean() * 100)

# Verify that among routes not in T-100, 100.0% of rows are null 
unmatched_all = full_spine[full_spine["ROUTE"].isin(missing_from_t100)]
print("\nNull % on routes missing from T-100:")
print(unmatched_all[["T100_Market_Pax_lag_12", "T100_Market_Pax_lag_24"]].isna().mean() * 100)


In [ ]:
base_df = full_spine.copy()
base_df.head()

In [ ]:
base_df.rename(columns={'weekend_peak_days':'WEEKEND', 'us_holiday_count':'HOLIDAY',
       'is_easter_month':'EASTER', 'is_covid':'COVID', 'T100_Market_Pax_lag_12':'MPAX_L12',
       'T100_Market_Pax_lag_24':'MPAX_L24', 'T100_Target_Pax_Share_lag_12':'PAX_SHARE_L12',
       'T100_Target_Pax_Share_lag_24':'PAX_SHARE_L24', 'T100_Carrier_Count_lag_12':'AIRLN_COUNT_L12',
       'T100_Carrier_Count_lag_24':'AIRLN_COUNT_L24'},inplace=True)
base_df.head()

In [ ]:
base_df = base_df.drop(columns=[col for col in base_df.columns if col.endswith('12')])

base_df[(base_df.DEP_YEAR == 2027) & (base_df.DEP_MONTH >1)].sample(5)

In [ ]:
base_df.sort_values(by='AIRLN_COUNT_L24',ascending=False)

In [ ]:
base_df.to_csv(DATA_DIR/'made/AA_MASTER_V1.csv',index=False)

---
# 3 Base_df + cargo + enplanements

## Cargo

In [ ]:

years = range(2017, 2026)
cargo_dfs = []

for year in years:
  df = pd.read_csv(DATA_DIR / f'Cargo{year}.csv')

  df.columns = df.columns.str.strip()

  # Find and match the column containing the current year and 'Landed Weight'
  weight_col = [
      col
      for col in df.columns
      if str(year) in col and (f'{year} landed\nweight (lbs.)' in col.lower() or f'{year} landed\r\nweight (lbs.)' in col.lower())
  ][0]

  # Rename to standard column name and tag the year
  df = df.rename(columns={weight_col: 'LANDED_WEIGHT'})
  df['YEAR'] = year

  

  df.drop(columns=[f'{year} Landed\nWeight (lbs.)', f'{year-1} Landed\nWeight (lbs.)', '% Change'], inplace=True, errors='ignore')
  cargo_dfs.append(df)


# Combine all years into one master DataFrame
Cargo = pd.concat(cargo_dfs, ignore_index=True)
Cargo.head()


In [ ]:
Cargo.drop(columns=['ST','Rank','ADO','RO','Airport Name','City','No.',
'FAA ADO','FAA Region'],inplace=True, errors='ignore')
print(Cargo.isnull().value_counts())
Cargo.head()


In [ ]:
Cargo.dropna(inplace=True)
print(Cargo.isnull().value_counts()) 
print(Cargo[Cargo.isnull().any(axis=1)])


In [ ]:
Cargo.rename(columns={'Locid':'LOCID','Svc Lvl':'SVC_LVL','Hub':'HUB'},inplace=True)
Cargo.to_csv(DATA_DIR / 'made/Cargo.csv', index=False)


In [ ]:

# Standardize column names & ensure strictly 1 row per ['Airport', 'Year']
cargo_clean = (
    Cargo.rename(columns={
        "LOCID": "Airport",
        "YEAR": "Year",
        "LANDED_WEIGHT": "Cargo_Landed_Weight",
        "SVC_LVL": "Airport_Svc_Level",
        "HUB": "Airport_Hub_Size",
    })
    .sort_values("Cargo_Landed_Weight", ascending=False)
    .groupby(["Airport", "Year"], as_index=False)
    .agg(
        Cargo_Landed_Weight=("Cargo_Landed_Weight", "sum"),
        Airport_Svc_Level=("Airport_Svc_Level", "first"),
        Airport_Hub_Size=("Airport_Hub_Size", "first"),
    )
)

# Lag Landed Weight by 1 and 2 years 
# Build a continuous yearly grid per Airport out to 2028 so 2027-2028 get lagged values
all_years = pd.DataFrame({"Year": range(cargo_clean["Year"].min(), 2029)})
airports = cargo_clean[["Airport"]].drop_duplicates()

cargo_grid = airports.merge(all_years, how="cross").merge(
    cargo_clean, on=["Airport", "Year"], how="left"
)

# Fill historical missing years (<= max year in cargo_df) with 0
max_cargo_year = cargo_clean["Year"].max()
cargo_grid.loc[cargo_grid["Year"] <= max_cargo_year, "Cargo_Landed_Weight"] = (
    cargo_grid.loc[cargo_grid["Year"] <= max_cargo_year, "Cargo_Landed_Weight"].fillna(0)
)

# Forward-fill static categorical tiers (svc level & hub size) into 2027-2028
cargo_grid = cargo_grid.sort_values(["Airport", "Year"]).reset_index(drop=True)
cargo_grid[["Airport_Svc_Level", "Airport_Hub_Size"]] = (
    cargo_grid.groupby("Airport")[["Airport_Svc_Level", "Airport_Hub_Size"]].ffill()
)

# Create 1-year and 2-year lags of Landed Weight, then drop raw unlagged weight
cargo_grid["Cargo_Landed_Weight_lag_1yr"] = cargo_grid.groupby("Airport")["Cargo_Landed_Weight"].shift(1)
cargo_grid["Cargo_Landed_Weight_lag_2yr"] = cargo_grid.groupby("Airport")["Cargo_Landed_Weight"].shift(2)
cargo_ready = cargo_grid.drop(columns=["Cargo_Landed_Weight"])
cargo_ready.sample(5)



In [ ]:
full_spine = base_df.copy()
full_spine.head()


In [ ]:



latest_cargo_year = int(cargo_clean["Year"].max())  # 2025
full_spine["Cargo_Lookup_Year"] = (full_spine["DEP_YEAR"] - 2).clip(upper=latest_cargo_year)
cargo_clean.rename(columns={
        "Cargo_Landed_Weight": "CARGO_WGT",
        "Airport_Svc_Level": "SVC_LVL",
        "Airport_Hub_Size": "HUB",
    },inplace=True)

# Double Left-Join cargo_clean directly onto full_spine 
full_spine1 = left_join_feature_table(
    spine=full_spine,
    feature_df=cargo_clean,
    left_keys=[["ORG_AIRPRT", "Cargo_Lookup_Year"], ["DES_AIRPRT", "Cargo_Lookup_Year"]],
    right_keys=["Airport", "Year"],
    prefixes=["ORG_", "DES_"],
)

full_spine1 = full_spine1.drop(columns=["Cargo_Lookup_Year"])


In [ ]:
pd.set_option('display.float_format', '{:,.2f}'.format)
full_spine1.head()

In [ ]:
full_spine1.loc[:, full_spine1.columns.str.endswith('WGT')].isnull().value_counts()


In [ ]:
full_spine1.loc[:, full_spine1.columns.str.endswith('LVL')].isnull().value_counts()


In [ ]:
full_spine1.loc[:, full_spine1.columns.str.endswith('HUB')].isnull().value_counts()


In [ ]:
base_df = full_spine1
base_df.head()


In [ ]:
base_df.to_csv(DATA_DIR / 'made/AA_MASTER_V2.csv')

To be Continued in part 2...
